In [12]:
import polars as pl
from pathlib import Path
from urllib.request import urlopen, Request
from tqdm import tqdm

In [13]:

out_dir = Path("/mnt/Fast2T/data/common_crawl/")

urls = (
    pl.read_csv("/mnt/Fast2T/kotlin/2026/data-science-project/data/paths")
    .with_columns(url="https://data.commoncrawl.org/" + pl.col("url"))
    .sample(100)
    .get_column("url")
    .to_list()
)
CHUNK = 1 << 20

for i, url in enumerate(urls):
    dest = out_dir / Path(url).name
    if dest.exists():
        continue

    resp = urlopen(Request(url))
    total = int(resp.headers.get("Content-Length", 0)) or None

    with (
        open(dest, "wb") as f,
        tqdm(
            total=total,
            unit="B",
            unit_scale=True,
            desc=f"[{i + 1}/{len(urls)}] {dest.name[:40]}",
        ) as bar,
    ):
        while chunk := resp.read(CHUNK):
            f.write(chunk)
            bar.update(len(chunk))

[1/100] CC-MAIN-20190426133444-20190426155119-00: 100%|██████████| 870M/870M [00:20<00:00, 41.6MB/s] 
[2/100] CC-MAIN-20190418121317-20190418143317-00: 100%|██████████| 875M/875M [00:21<00:00, 41.1MB/s] 
[3/100] CC-MAIN-20190420020937-20190420042937-00: 100%|██████████| 906M/906M [00:21<00:00, 41.2MB/s] 
[4/100] CC-MAIN-20190424094510-20190424120510-00: 100%|██████████| 875M/875M [00:23<00:00, 37.3MB/s] 
[5/100] CC-MAIN-20190422095521-20190422121521-00: 100%|██████████| 863M/863M [00:20<00:00, 41.4MB/s] 
[6/100] CC-MAIN-20190421100217-20190421122217-00: 100%|██████████| 878M/878M [00:21<00:00, 40.7MB/s] 
[7/100] CC-MAIN-20190423154842-20190423180842-00: 100%|██████████| 859M/859M [00:21<00:00, 39.5MB/s] 
[8/100] CC-MAIN-20190425114058-20190425140058-00: 100%|██████████| 879M/879M [00:21<00:00, 41.3MB/s] 
[9/100] CC-MAIN-20190426133444-20190426155444-00: 100%|██████████| 861M/861M [00:21<00:00, 39.8MB/s] 
[10/100] CC-MAIN-20190426073513-20190426095513-00: 100%|██████████| 876M/876M [00:

In [ ]:
import gzip
import re
from pathlib import Path

import polars as pl
import pyarrow.parquet as pq
from warcio.archiveiterator import ArchiveIterator

_SURROGATES = re.compile(f"[{chr(0xD800)}-{chr(0xDFFF)}]")


def sanitize_utf8(s: str) -> str:
    if not s:
        return s
    s = _SURROGATES.sub("\ufffd", s)
    return s.encode("utf-8", "replace").decode("utf-8")


_charset_re = re.compile(r"charset\s*=\s*([^\s;]+)", re.IGNORECASE)


def charset_from_content_type(ct: str) -> str | None:
    if not ct:
        return None
    m = _charset_re.search(ct)
    return m.group(1).strip("\"'") if m else None


def decode_html(body: bytes, charset: str | None) -> str:
    if not body:
        return ""
    enc = (charset or "utf-8").strip()
    try:
        text = body.decode(enc, errors="replace")
    except (LookupError, UnicodeDecodeError):
        text = body.decode("utf-8", errors="replace")
    return sanitize_utf8(text)


def warc_to_single_parquet_html_only(
        warc_paths: list[Path],
        out_file: Path,
        chunk_size: int = 20_000,
        zstd_level: int | None = None,  # optional
        require_status_200: bool = False  # optional
) -> None:
    out_file.parent.mkdir(parents=True, exist_ok=True)
    if out_file.exists():
        out_file.unlink()  # avoid accidental append/overwrite ambiguity

    contents: list[str] = []
    writer: pq.ParquetWriter | None = None

    def flush() -> None:
        nonlocal contents, writer
        if not contents:
            return

        n = len(contents)
        df = df = pl.DataFrame(
            {
                "content": pl.Series("content", [sanitize_utf8(x) for x in contents], dtype=pl.Utf8),
            }
        )

        table = df.to_arrow()

        if writer is None:
            writer = pq.ParquetWriter(
                where=str(out_file),
                schema=table.schema,
                compression="zstd",
                compression_level=zstd_level,
                use_dictionary=True,
                write_statistics=False
            )

        writer.write_table(table)
        contents = []

    try:
        for warc_path in warc_paths:
            with gzip.open(warc_path, "rb") as f:
                for record in ArchiveIterator(f):
                    if getattr(record, "rec_type", None) != "response":
                        continue

                    hh = getattr(record, "http_headers", None)
                    if not hh:
                        continue

                    ct = (hh.get_header("Content-Type") or "")
                    if "text/html" not in ct.lower():
                        continue

                    if require_status_200:
                        try:
                            if int(hh.get_statuscode() or 0) != 200:
                                continue
                        except Exception:
                            continue

                    body_bytes = record.content_stream().read()
                    html = decode_html(body_bytes, charset_from_content_type(ct))
                    if not html:
                        continue

                    contents.append(sanitize_utf8(html))
                    if len(contents) >= chunk_size:
                        flush()

        flush()
    finally:
        if writer is not None:
            writer.close()


for file in Path("/mnt/Fast2T/data/common_crawl/").iterdir():
    if not file.name.endswith(".warc.gz"):
        continue

    print(file)
    warc_to_single_parquet_html_only(
        warc_paths=[file],
        out_file=Path("/mnt/Fast2T/data/common_data_parquet/") / file.name.replace(".warc.gz", ".parquet"),
    )